In [ ]:
import random

import polars as pl

from rich.pretty import pprint
from rich.markdown import Markdown
from more_itertools import unique_everseen

# from por.llm_agents import LyricsValidator

In [ ]:
ARTIST_MAP = {
    "Taylor Swift": "Taylor Swift",
    "The Doors": "The Doors",
    "Bjrk": "Björk",
    "David Bowie": "David Bowie",
    "The Smiths": "The Smiths",
    "Marika Hackman": "Marika Hackman",
    "The D": "The Dø",
    "The Growlers": "The Growlers",
    "Portishead": "Portishead",
    "The Buttertones": "The Buttertones",
    "PJ Harvey": "PJ Harvey",
    "Massive Attack": "Massive Attack",
    "Gorillaz": "Gorillaz",
    "Ramones": "Ramones",
    "Gustavo Cerati": "Gustavo Cerati",
    "Arcade Fire": "Arcade Fire",
    "Charly Garca": "Charly García",
    "Dua Lipa": "Dua Lipa",
    "The Beatles": "The Beatles",
    "Queen": "Queen",
    "Talking Heads": "Talking Heads",
    "Radiohead": "Radiohead",
    "Madonna": "Madonna",
    "The Rolling Stones": "The Rolling Stones",
    "Elvis Presley": "Elvis Presley",
    "Yeah Yeah Yeahs": "Yeah Yeah Yeahs",
    "The Cure": "The Cure",
    "Soda Stereo": "Soda Stereo",
    "The Sound": "The Sound",
    # "Lowlife": "Lowlife",
    "Metric": "Metric",
    "Marilina Bertoldi": "Marilina Bertoldi",
    "Sumo": "Sumo",
    "Modern Eon": "Modern Eon",
    "Cranes": "Cranes",
    # "Virus": "Virus",
    "The Strokes": "The Strokes",
    "Blonde Redhead": "Blonde Redhead",
    "Justice": "Justice",
    "Daft Punk": "Daft Punk",
    "MGMT": "MGMT",
    "Ariel Pink": "Ariel Pink",
    "John Maus": "John Maus",
    "Mystic Braves": "Mystic Braves",
    "Depeche Mode": "Depeche Mode",
    "Duran Duran": "Duran Duran",
    "Travis": "Travis",
    "Metronomy": "Metronomy",
    "Isaac Delusion": "Isaac Delusion",
    "Alexandra Savior": "Alexandra Savior",
    "Morrissey": "Morrissey",
    "Beck": "Beck",
    "Lou Reed": "Lou Reed",
    "Allah-Las": "Allah-Las",
    "AURORA": "AURORA",
    "Franz Ferdinand": "Franz Ferdinand",
    "Patricio Rey y sus Redonditos de Ricota": "Patricio Rey y sus Redonditos de Ricota",
    "Los Fundamentalistas del Aire Acondicionado": "Los Fundamentalistas del Aire Acondicionado",
}

In [ ]:
reader = pl.read_csv_batched(
    source="/resources/datasets/song_lyrics.csv",
    n_threads=4,
)

lyrics = []
while True:
    batch = reader.next_batches(1)
    if batch is None:
        break

    for df in batch:
        data_items = df.to_dicts()
        for data_item in data_items:
            if data_item["artist"] in ARTIST_MAP:
                lyrics.append(
                    data_item
                    | {
                        "artist": ARTIST_MAP[data_item["artist"]],
                    }
                )

print(f"lyrics => {len(lyrics)}")

In [ ]:
diff = set(ARTIST_MAP.values()) - set(li["artist"] for li in lyrics)
assert not diff
# pprint(lyrics[:3])

In [ ]:
lala = [
    item
    for item in lyrics
    if item["artist"] == "Patricio Rey y sus Redonditos de Ricota"
]
len(lala)

In [ ]:
pprint(lala)

In [ ]:
unique_lyrics = unique_everseen(
    lyrics,
    key=lambda ly: (
        ly["title"],
        ly["artist"],
        ly["year"],
    ),
)

unique_lyrics = list(unique_lyrics)
print(f"unique_lyrics: {len(unique_lyrics)}")

In [ ]:
user_prompts = [
    f"# Artist\n{ly['artist']}\n\n# Title\n{ly['title']}\n\n# Lyrics\n{ly['lyrics']}"
    for ly in lyrics
]

In [ ]:
prompt = random.choice(user_prompts)
Markdown(prompt)

In [ ]:
lv = LyricsValidator(
    max_concurrency=20,
    cache=RedisCache(),
)

lv_outputs = await lv.batch_generate(user_prompts=user_prompts)

In [ ]:
assert len(lyrics) == len(lv_outputs)
valid_lyrics = [
    ly | {"language": lvo.language}
    for ly, lvo in zip(lyrics, lv_outputs)
    if lvo.is_valid
]
print(f"valid_lyrics: {len(valid_lyrics)}")

In [ ]:
file_path = "/resources/documents/lyrics/lyrics.json"
save_json(obj=valid_lyrics, file_path=file_path)